# S4.1.1 — Zero / near-zero distributions

**Purpose:** Show how values pile up, how close they sit to zero, and where zeros fall in pair space. This notebook does **not** classify relationships and does **not** apply 5% / 15% structure labels. Counts and percentages are for reading.

Three views, same 5 models × 6 zones:

1. **Value histogram** (log count) — including a spike at exact 0 if it exists.
2. **`log10|v|`** for variables that can be zero — how near non-zero values get, plus the exact-0 count on the side.
3. **Pair scatter** — grey continuous, purple exact 0, orange near-zero (`0 < |v| ≤ 10^{-4}`).

Maps of where zeros sit on Earth stay in **S4.0**. Classifier A/B stays in **S4.1**. Continuous scatter/LOWESS stays in **S4.2**.

Near-zero cut `EPS_NEAR = 1e-4` matches the continuous filter used in S4.1 block B.


In [ ]:
from __future__ import annotations

import warnings
from pathlib import Path

import matplotlib
if str(matplotlib.get_backend()).lower() == "agg":
    try:
        matplotlib.use("module://matplotlib_inline.backend_inline", force=True)
    except Exception:
        pass

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
from IPython.display import Image, display

warnings.filterwarnings("ignore", category=FutureWarning)

plt.rcParams.update({
    "figure.dpi": 120, "savefig.dpi": 160,
    "figure.facecolor": "white", "axes.facecolor": "white",
    "font.family": "DejaVu Sans", "axes.edgecolor": "#30343B",
    "axes.linewidth": 0.8,
})


def locate_case_dir() -> Path:
    for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (candidate / "cmip_utils.py").exists() and candidate.name == "caseA":
            return candidate
        nested = candidate / "case" / "caseA"
        if (nested / "cmip_utils.py").exists():
            return nested
    raise FileNotFoundError("Could not locate case/caseA/")


CASE_DIR = locate_case_dir()
DATA_ROOT = Path("/Volumes/mimi-T9/CMIP6")

MODELS = ["CESM2", "CNRM-CM6-1", "CanESM5", "GFDL-CM4", "CMCC-CM2-SR5"]
ZONES = ["all_land", "WW", "WD", "CW", "CD", "LI"]
ZONE_SHORT = {
    "all_land": "All land",
    "WW": "Wet-warm (WW)",
    "WD": "Dry-warm (WD)",
    "CW": "Wet-cold (CW)",
    "CD": "Dry-cold (CD)",
    "LI": "Land ice (LI)",
}

VARIABLE_PAIRS = [
    ("P", "Q", "P → Q"),
    ("ET", "Q", "ET → Q"),
    ("mrros", "Q", "mrros → Q"),
    ("prsn", "Q", "prsn → Q"),
    ("tran", "Q", "tran → Q"),
    ("evspsblsoi", "Q", "evspsblsoi → Q"),
    ("hfls", "Q", "hfls → Q"),
    ("hfss", "Q", "hfss → Q"),
    ("lai", "Q", "lai → Q"),
    ("tas", "Q", "tas → Q"),
    ("rsds", "Q", "rsds → Q"),
    ("mrso", "Q", "mrso → Q"),
    ("mrsos", "Q", "mrsos → Q"),
    ("rlds", "Q", "rlds → Q"),
    ("rlus", "Q", "rlus → Q"),
    ("rsus", "Q", "rsus → Q"),
    ("P", "ET", "P → ET"),
]

ALL_VARS = list(dict.fromkeys(v for x, y, _ in VARIABLE_PAIRS for v in (x, y)))
# Variables that can physically or numerically sit at zero.
NEARZERO_VARS = ["Q", "ET", "mrros", "prsn", "tran", "evspsblsoi", "lai", "mrso", "mrsos"]

START_YEAR, END_YEAR = 1985, 2014
EPS_NEAR = 1e-4  # same cut as S4.1 block B
# Cumulative |v| thresholds shown in the inventory (exact 0 is separate).
MAG_CUTS = [
    (1e-6, '1e-6'),
    (1e-4, '1e-4'),
    (1e-2, '1e-2'),
    (1.0, '1'),
    (10.0, '10'),
    (100.0, '100'),
]

PLOT_DIST = True
PLOT_NEARZERO = True
PLOT_SCATTER = True
SHOW_INLINE = True

OUTPUT_DIR = CASE_DIR / "output" / "S4.1.1"
DIST_DIR = OUTPUT_DIR / "dist"
NEAR_DIR = OUTPUT_DIR / "nearzero"
SCATTER_DIR = OUTPUT_DIR / "scatter"
for d in (OUTPUT_DIR, DIST_DIR, NEAR_DIR, SCATTER_DIR):
    d.mkdir(parents=True, exist_ok=True)

print(f"Case dir:   {CASE_DIR}")
print(f"EPS_NEAR:   {EPS_NEAR:g}")
print(f"Output:     {OUTPUT_DIR}")
print(f"Variables:  {ALL_VARS}")


In [ ]:
# Load zone climatology (same tables as S4.0 / S4.1).
runs = []
for model in MODELS:
    search_roots = [
        DATA_ROOT / model / "historical",
        CASE_DIR / "data" / model / "historical",
    ]
    zone_name = f"zone_climatology_{START_YEAR}_{END_YEAR}.parquet"
    found = False
    for root in search_roots:
        if not root.exists():
            continue
        paths = sorted(root.glob(f"*/*/land/zones/{zone_name}"))
        if paths:
            table = pd.read_parquet(paths[0])
            table.rename(columns={"R": "Q"}, inplace=True)
            runs.append({"model": model, "data": table, "path": paths[0]})
            print(f"{model}: {len(table):,} rows — {paths[0]}")
            found = True
            break
    if not found:
        print(f"{model}: SKIPPED")

if not runs:
    raise FileNotFoundError("No CMIP6 zone-climatology tables were found. Mount /Volumes/mimi-T9.")
print(f"Loaded {len(runs)} models")


In [ ]:
UNIT_LABELS = {
    "P": "Precipitation (mm/yr)",
    "ET": "Evapotranspiration (mm/yr)",
    "Q": "Total runoff (mm/yr)",
    "hfls": "Latent heat flux (W/m²)",
    "hfss": "Sensible heat flux (W/m²)",
    "tran": "Transpiration (mm/yr)",
    "evspsblsoi": "Soil evaporation (mm/yr)",
    "mrros": "Surface runoff (mm/yr)",
    "mrso": "Total soil moisture (kg/m²)",
    "mrsos": "Topsoil moisture (kg/m²)",
    "lai": "LAI (m²/m²)",
    "tas": "Temperature (K)",
    "prsn": "Snowfall (mm/yr)",
    "rlds": "Downward LW (W/m²)",
    "rlus": "Upward LW (W/m²)",
    "rsds": "Downward SW (W/m²)",
    "rsus": "Upward SW (W/m²)",
}


def get_var_values(run, var, zone):
    data = run["data"]
    if var not in data.columns:
        return None
    sub = data if zone == "all_land" else data[data["analysis_zone"] == zone]
    vals = sub[var].to_numpy(dtype=float)
    vals = vals[np.isfinite(vals)]
    return vals if len(vals) else None


def get_xy(run, x_var, y_var, zone):
    data = run["data"]
    if x_var not in data.columns or y_var not in data.columns:
        return None, None
    sub = data if zone == "all_land" else data[data["analysis_zone"] == zone]
    x = sub[x_var].to_numpy(dtype=float)
    y = sub[y_var].to_numpy(dtype=float)
    finite = np.isfinite(x) & np.isfinite(y)
    if finite.sum() < 5:
        return None, None
    return x[finite], y[finite]


def style_axes(ax):
    for spine in ax.spines.values():
        spine.set_color("#30343B")
        spine.set_linewidth(1.0)
    ax.tick_params(direction="out", length=3.5, width=0.8, labelsize=8)
    ax.grid(True, color="#D3D3D3", linewidth=0.5, alpha=0.7, zorder=0)


def annotate(ax, lines, fontsize=7.2):
    ax.text(
        0.97, 0.97, "\n".join(lines),
        transform=ax.transAxes, fontsize=fontsize, va="top", ha="right",
        family="DejaVu Sans",
        bbox=dict(boxstyle="round,pad=0.22", facecolor="white",
                  alpha=0.88, edgecolor="#CCCCCC", linewidth=0.6),
        zorder=10,
    )


def panel_headers(ax, row_i, col_j, model, xlabel, ylabel="Count"):
    if row_i == 0:
        ax.set_title(ZONE_SHORT[ZONES[col_j]], fontsize=12, fontweight="semibold", pad=5)
    if col_j == 0:
        ax.set_ylabel(ylabel, fontsize=9)
        ax.text(
            -0.38, 0.5, model, transform=ax.transAxes,
            fontsize=11, fontweight="bold", ha="right", va="center", rotation=90,
        )
    if row_i == len(MODELS) - 1:
        ax.set_xlabel(xlabel, fontsize=9)


def save_show(fig, path):
    fig.savefig(path, dpi=160, bbox_inches="tight")
    print(f"  Saved: {path.name}")
    if SHOW_INLINE:
        display(Image(filename=str(path)))
    plt.close(fig)


COL_W, ROW_H = 3.2, 2.8
print("Helpers ready.")


In [ ]:
# Inventory: exact 0 and cumulative |v| ≤ cut. One row per variable × model × zone.
# Not a median across models, and not a 5%/15% structure label.

def magnitude_counts(vals):
    vals = np.asarray(vals, float)
    n = len(vals)
    abs_v = np.abs(vals)
    n_ez = int(np.sum(vals == 0.0))
    out = {
        "N": n,
        "n_eq_0": n_ez,
        "pct_eq_0": 100.0 * n_ez / n,
    }
    for cut, key in MAG_CUTS:
        n_le = int(np.sum(abs_v <= cut))
        out[f"n_le_{key}"] = n_le
        out[f"pct_le_{key}"] = 100.0 * n_le / n
    nz = abs_v[vals != 0.0]
    out["min_abs_nonzero"] = float(nz.min()) if len(nz) else np.nan
    return out


var_rows = []
for run in runs:
    model = run["model"]
    for var in ALL_VARS:
        for zone in ZONES:
            vals = get_var_values(run, var, zone)
            if vals is None:
                continue
            rec = {"variable": var, "Model": model, "Zone": zone}
            rec.update(magnitude_counts(vals))
            var_rows.append(rec)

var_df = pd.DataFrame(var_rows)
var_path = OUTPUT_DIR / "zero_variable_inventory.csv"
var_df.to_csv(var_path, index=False)

pct_cols = ["pct_eq_0"] + [f"pct_le_{key}" for _, key in MAG_CUTS]


def show_zone(zone, title):
    sub = var_df[var_df["Zone"] == zone].copy()
    print(f"\n=== {title} ===")
    print("Columns are cumulative: exact 0, then |v| ≤ 1e-6, 1e-4, 1e-2, 1, 10, 100.")
    print("Each row is one model. Units differ (K, W/m², mm/yr), so large cuts are uninformative for tas/radiation.\n")
    for var in ALL_VARS:
        block = sub[sub["variable"] == var][["Model"] + pct_cols].set_index("Model")
        if block.empty:
            continue
        # Skip all-zero tables (no mass near 0 at any listed cut except maybe 100)
        if (block["pct_eq_0"].max() < 0.05) and (block["pct_le_1e-4"].max() < 0.05):
            continue
        print(f"— {var}")
        display(block.round(1))


show_zone("all_land", "all_land  |  % of finite cells")
show_zone("WW", "WW")
show_zone("LI", "LI")

# Pair table: exact-0 and near-0 on either axis, still per model (all_land).
pair_rows = []
for run in runs:
    model = run["model"]
    for x_var, y_var, pair_label in VARIABLE_PAIRS:
        for zone in ZONES:
            x, y = get_xy(run, x_var, y_var, zone)
            if x is None:
                continue
            n = len(x)
            xz, yz = x == 0.0, y == 0.0
            n_p00 = int(np.sum(xz & yz))
            n_p01 = int(np.sum(xz & ~yz))
            n_p10 = int(np.sum(~xz & yz))
            n_ez = n_p00 + n_p01 + n_p10
            either_le = (np.abs(x) <= EPS_NEAR) | (np.abs(y) <= EPS_NEAR)
            pair_rows.append({
                "Pair": pair_label, "x_var": x_var, "y_var": y_var,
                "Model": model, "Zone": zone, "N": n,
                "n_both_zero": n_p00, "n_x_zero": n_p01, "n_y_zero": n_p10,
                "pct_eq_0": 100.0 * n_ez / n,
                "pct_x_eq_0": 100.0 * (n_p00 + n_p01) / n,
                "pct_y_eq_0": 100.0 * (n_p00 + n_p10) / n,
                "pct_either_le_1e-4": 100.0 * either_le.mean(),
            })

pair_df = pd.DataFrame(pair_rows)
pair_path = OUTPUT_DIR / "zero_pair_inventory.csv"
pair_df.to_csv(pair_path, index=False)

print("\n=== Pair space, all_land: exact 0 on x or y, and either |v| ≤ 1e-4 ===")
pl = pair_df[pair_df["Zone"] == "all_land"]
display(
    pl.pivot_table(index="Pair", columns="Model", values="pct_eq_0").round(1)
)

print(f"\nSaved {var_path.name} ({len(var_df)} rows)")
print(f"Saved {pair_path.name} ({len(pair_df)} rows)")


## 1. Value distributions

One 5×6 figure per variable. Blue histogram of all finite values, log count axis, dashed red line at 0. The box is **N**, exact-zero count and %, and `min|v|` among non-zeros. No screening threshold.


In [ ]:
if not PLOT_DIST:
    print("PLOT_DIST=False")
else:
    n_rows, n_cols = len(MODELS), len(ZONES)
    for var in ALL_VARS:
        fig, axes = plt.subplots(
            n_rows, n_cols,
            figsize=(COL_W * n_cols, ROW_H * n_rows),
            constrained_layout=True, squeeze=False,
        )
        fig.suptitle(
            f"S4.1.1 value distribution — {UNIT_LABELS.get(var, var)}",
            fontsize=15, fontweight="semibold",
        )
        for row_i, run in enumerate(runs):
            model = run["model"]
            for col_j, zone in enumerate(ZONES):
                ax = axes[row_i, col_j]
                vals = get_var_values(run, var, zone)
                if vals is None or len(vals) < 5:
                    ax.text(0.5, 0.5, "no data", ha="center", va="center",
                            transform=ax.transAxes, color="#999")
                    style_axes(ax)
                    panel_headers(ax, row_i, col_j, model, UNIT_LABELS.get(var, var))
                    continue
                n = len(vals)
                n_ez = int(np.sum(vals == 0.0))
                n_near = int(np.sum((vals != 0.0) & (np.abs(vals) <= EPS_NEAR)))
                n_bins = min(80, max(30, n // 50))
                ax.hist(vals, bins=n_bins, color="#4A7FB5", edgecolor="#2A5F95",
                        linewidth=0.25, alpha=0.88, zorder=2)
                ax.set_yscale("log")
                ax.axvline(0.0, color="#C02020", lw=1.0, ls="--", alpha=0.75, zorder=3)
                abs_nz = np.abs(vals[vals != 0.0])
                lines = [f"N={n}", f"ez={n_ez} ({100*n_ez/n:.1f}%)"]
                if n_near:
                    lines.append(f"near={n_near} ({100*n_near/n:.1f}%)")
                if len(abs_nz):
                    lines.append(f"min|v|={abs_nz.min():.1e}")
                annotate(ax, lines)
                style_axes(ax)
                panel_headers(ax, row_i, col_j, model, UNIT_LABELS.get(var, var))
        save_show(fig, DIST_DIR / f"dist_{var}.png")
    print(f"Distribution figures → {DIST_DIR}")


## 2. How close to zero (`log10|v|`)

Exact zeros cannot sit on a log axis, so they are counted in the box, not in the bars. Non-zero `|v|` is histogrammed on `log10`. The orange line is `EPS_NEAR = 10^{-4}` (S4.1 continuous cut). Read `ez%` and `|v|≤1e-4%` yourself; nothing is auto-labelled as “along-axis”.


In [ ]:
if not PLOT_NEARZERO:
    print("PLOT_NEARZERO=False")
else:
    n_rows, n_cols = len(MODELS), len(ZONES)
    for var in NEARZERO_VARS:
        fig, axes = plt.subplots(
            n_rows, n_cols,
            figsize=(COL_W * n_cols, ROW_H * n_rows),
            constrained_layout=True, squeeze=False,
        )
        fig.suptitle(
            f"S4.1.1 near-zero — {UNIT_LABELS.get(var, var)}   |   log₁₀|v|  (v ≠ 0)",
            fontsize=15, fontweight="semibold",
        )
        for row_i, run in enumerate(runs):
            model = run["model"]
            for col_j, zone in enumerate(ZONES):
                ax = axes[row_i, col_j]
                vals = get_var_values(run, var, zone)
                if vals is None or len(vals) < 5:
                    ax.text(0.5, 0.5, "no data", ha="center", va="center",
                            transform=ax.transAxes, color="#999")
                    style_axes(ax)
                    panel_headers(ax, row_i, col_j, model, f"log₁₀|{var}|")
                    continue
                n = len(vals)
                n_ez = int(np.sum(vals == 0.0))
                abs_nz = np.abs(vals[vals != 0.0])
                n_le = int(np.sum(np.abs(vals) <= EPS_NEAR))
                if len(abs_nz) < 3:
                    ax.text(0.5, 0.5, f"all zeros\n({n_ez}/{n})",
                            ha="center", va="center", color="#C03030",
                            transform=ax.transAxes)
                    style_axes(ax)
                    panel_headers(ax, row_i, col_j, model, f"log₁₀|{var}|")
                    continue
                log_abs = np.log10(abs_nz)
                n_bins = min(70, max(25, len(log_abs) // 40))
                ax.hist(log_abs, bins=n_bins, color="#4A7FB5", edgecolor="#2A5F95",
                        linewidth=0.25, alpha=0.88, zorder=2)
                ax.set_yscale("log")
                ax.axvline(np.log10(EPS_NEAR), color="#E07A00", lw=1.2, ls="--",
                           zorder=3)
                lines = [
                    f"N={n}",
                    f"ez={n_ez} ({100*n_ez/n:.1f}%)",
                    f"|v|≤1e-4: {n_le} ({100*n_le/n:.1f}%)",
                    f"min|v|={abs_nz.min():.1e}",
                ]
                annotate(ax, lines, fontsize=6.8)
                style_axes(ax)
                panel_headers(ax, row_i, col_j, model, f"log₁₀|{var}|")
        from matplotlib.lines import Line2D
        fig.legend(
            handles=[Line2D([0], [0], color="#E07A00", ls="--", lw=1.2,
                            label=f"EPS_NEAR = {EPS_NEAR:g}")],
            loc="lower center", ncol=1, fontsize=10, frameon=False,
            bbox_to_anchor=(0.5, -0.02),
        )
        save_show(fig, NEAR_DIR / f"nearzero_{var}.png")
    print(f"Near-zero figures → {NEAR_DIR}")


## 3. Pair scatter — where zeros sit

Grey: both `|v| > 10^{-4}`. Purple: exact 0 on x and/or y (the axis wall or the origin). Orange: near-zero on at least one axis, neither coordinate exact 0. Shared column limits across models. Box: N, exact-zero %, near-zero %, and the split `x=0` / `y=0` / both.


In [ ]:
SCATTER_COLORS = {
    "cont": "#555555",
    "near": "#E07A00",
    "exact": "#6B21A8",
}


def pair_masks(x, y):
    xz, yz = x == 0.0, y == 0.0
    exact = xz | yz
    near = (
        ((np.abs(x) <= EPS_NEAR) | (np.abs(y) <= EPS_NEAR))
        & ~exact
    )
    cont = ~exact & ~near
    return {"cont": cont, "near": near, "exact": exact}


def column_limits(x_var, y_var, zone):
    xs, ys = [], []
    for run in runs:
        x, y = get_xy(run, x_var, y_var, zone)
        if x is None:
            continue
        xs.append(x)
        ys.append(y)
    if not xs:
        return (-1, 1), (-1, 1)

    def lim(a):
        a = np.concatenate(a)
        lo, hi = float(np.min(a)), float(np.max(a))
        pad = 0.04 * (hi - lo if hi > lo else max(abs(lo), 1.0))
        return lo - pad, hi + pad

    return lim(xs), lim(ys)


if not PLOT_SCATTER:
    print("PLOT_SCATTER=False")
else:
    n_rows, n_cols = len(MODELS), len(ZONES)
    for x_var, y_var, pair_label in VARIABLE_PAIRS:
        col_lims = {z: column_limits(x_var, y_var, z) for z in ZONES}
        fig, axes = plt.subplots(
            n_rows, n_cols,
            figsize=(COL_W * n_cols, ROW_H * n_rows),
            constrained_layout=True, squeeze=False,
        )
        fig.suptitle(
            f"S4.1.1 zero location — {pair_label}   (|v|≤{EPS_NEAR:g} = near)",
            fontsize=15, fontweight="semibold",
        )
        for row_i, run in enumerate(runs):
            model = run["model"]
            for col_j, zone in enumerate(ZONES):
                ax = axes[row_i, col_j]
                xlim, ylim = col_lims[zone]
                x, y = get_xy(run, x_var, y_var, zone)
                if x is None:
                    ax.text(0.5, 0.5, "no data", ha="center", va="center",
                            transform=ax.transAxes, color="#999")
                    style_axes(ax)
                    panel_headers(ax, row_i, col_j, model,
                                  UNIT_LABELS.get(x_var, x_var),
                                  UNIT_LABELS.get(y_var, y_var))
                    ax.set_xlim(*xlim)
                    ax.set_ylim(*ylim)
                    continue
                masks = pair_masks(x, y)
                n = len(x)
                n_ez = int(masks["exact"].sum())
                n_near = int(masks["near"].sum())
                xz, yz = x == 0.0, y == 0.0
                n_p00 = int(np.sum(xz & yz))
                n_p01 = int(np.sum(xz & ~yz))
                n_p10 = int(np.sum(~xz & yz))
                for key, s, a, z in (
                    ("cont", 1.2, 0.35, 1),
                    ("near", 6.0, 0.85, 3),
                    ("exact", 7.0, 0.88, 4),
                ):
                    m = masks[key]
                    if m.any():
                        ax.scatter(
                            x[m], y[m], s=s, c=SCATTER_COLORS[key],
                            alpha=a, linewidths=0, rasterized=True, zorder=z,
                        )
                ax.axvline(0.0, color="#AAAAAA", lw=0.6, ls=":", zorder=0)
                ax.axhline(0.0, color="#AAAAAA", lw=0.6, ls=":", zorder=0)
                lines = [
                    f"N={n}",
                    f"ez={n_ez} ({100*n_ez/n:.1f}%)",
                    f"near={n_near} ({100*n_near/n:.1f}%)",
                    f"x=0:{n_p01}  y=0:{n_p10}  both:{n_p00}",
                ]
                annotate(ax, lines, fontsize=6.6)
                style_axes(ax)
                ax.set_xlim(*xlim)
                ax.set_ylim(*ylim)
                panel_headers(
                    ax, row_i, col_j, model,
                    UNIT_LABELS.get(x_var, x_var),
                    UNIT_LABELS.get(y_var, y_var),
                )
        handles = [
            mpatches.Patch(color=SCATTER_COLORS["cont"], label="Continuous (|v| > 1e-4)"),
            mpatches.Patch(color=SCATTER_COLORS["near"], label="Near-zero (0 < |v| ≤ 1e-4)"),
            mpatches.Patch(color=SCATTER_COLORS["exact"], label="Exact zero (v = 0)"),
        ]
        fig.legend(
            handles=handles, loc="lower center", ncol=3, fontsize=10,
            frameon=False, bbox_to_anchor=(0.5, -0.03),
        )
        slug = f"{x_var}_{y_var}"
        save_show(fig, SCATTER_DIR / f"scatter_{slug}.png")
    print(f"Scatter figures → {SCATTER_DIR}")
